In [2]:
from typing import Literal
from pydantic import BaseModel, Field

In [63]:
class Triage(BaseModel):
    category: Literal["billing", "shipping", "other", "account"]
    priority: int = Field(ge=1, le=5)

In [65]:
import json

def ask_model(ticket_text):
    text = ticket_text.lower()
    if "charged" in text or "refund" in text:
        return '{"category":"billing","priority":4}'
    if "parcel" in text or "arrived" in text:
        return '{"category":"shipping", "priority":7}'
    if "password" in text:
        return '{"category":"account", "priority":2}'
    return "I am not sure how to sort this one"

In [66]:
answer = ask_model("Refund not recieved")
answer

'{"category":"billing","priority":4}'

In [67]:
answer = ask_model("I forgot my password")
answer

'{"category":"account", "priority":2}'

In [68]:
answer = ask_model("The parcel has not yet arrived")
answer

'{"category":"shipping", "priority":7}'

In [37]:
result = Triage.model_validate_json(answer)  # takes as string reads as json
result

ValidationError: 1 validation error for Triage
priority
  Input should be less than or equal to 5 [type=less_than_equal, input_value=7, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

In [10]:
result.priority

3

In [49]:
answer

'{"category:"account","priority":2}'

In [69]:
result = Triage.model_validate_json(ask_model("Forgot password"))
result

Triage(category='account', priority=2)

In [11]:
from pydantic import ValidationError

In [42]:
for answer in ["I m not sure how to do this", '{"category":"sales","priority":4}']:
    try:
        Triage.model_validate_json(answer)
    except ValidationError as e:
        print("----")
        print(e)
        print("----")

----
1 validation error for Triage
  Invalid JSON: expected ident at line 1 column 2 [type=json_invalid, input_value='I m not sure how to do this', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid
----
----
1 validation error for Triage
category
  Input should be 'billing', 'shipping' or 'other' [type=literal_error, input_value='sales', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
----


In [43]:
for answer in ["I m not sure how to do this", '{"category":"sales","priority":4}']:
    try:
        Triage.model_validate_json(answer)
    except ValidationError as e:
        print("----")
        print(e.errors())
        print("----")

----
[{'type': 'json_invalid', 'loc': (), 'msg': 'Invalid JSON: expected ident at line 1 column 2', 'input': 'I m not sure how to do this', 'ctx': {'error': 'expected ident at line 1 column 2'}, 'url': 'https://errors.pydantic.dev/2.14/v/json_invalid'}]
----
----
[{'type': 'literal_error', 'loc': ('category',), 'msg': "Input should be 'billing', 'shipping' or 'other'", 'input': 'sales', 'ctx': {'expected': "'billing', 'shipping' or 'other'"}, 'url': 'https://errors.pydantic.dev/2.14/v/literal_error'}]
----


In [21]:
tickets = [
  {
    "id": 1,
    "customer": "Asha",
    "text": "I was charged twice for one order"
  },
  {
    "id": 2,
    "customer": "Ben",
    "text": "My parcel has not arrived"
  },
  {
    "id": 3,
    "customer": "Chen",
    "text": "Can I get a refund for the blue mug?"
  },
  {
    "id": 4,
    "customer": "Dara",
    "text": "How do I change my password?"
  },
  {
    "id": 5,
    "customer": "Eli",
    "text": "The parcel arrived but the box was crushed"
  }
]

In [22]:
tickets

[{'id': '1', 'customer': 'Asha', 'text': 'I was charged twice for one order'},
 {'id': 2, 'customer': 'Ben', 'text': 'My parcel has not arrived'},
 {'id': 3, 'customer': 'Chen', 'text': 'Can I get a refund for the blue mug?'},
 {'id': 4, 'customer': 'Dara', 'text': 'How do I change my password?'},
 {'id': 5,
  'customer': 'Eli',
  'text': 'The parcel arrived but the box was crushed'}]

In [28]:
checked = []
for ticket in tickets:
    try:
        result = Triage.model_validate_json(ask_model(ticket["text"]))
    except ValidationError as e:
        print(e)
        print(f"{ticket["id"]} needs a person")
        continue
    print(ticket["id"], result.category, result.priority)
    checked.append(result)

print(len(checked), "checked")

1 billing 4
2 shipping 3
3 billing 4
1 validation error for Triage
  Invalid JSON: expected ident at line 1 column 2 [type=json_invalid, input_value='I am not sure how to sort this one', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid
4 needs a person
5 shipping 3
4 checked


In [34]:
print(json.dumps(Triage.model_json_schema(), indent=2))

{
  "properties": {
    "category": {
      "enum": [
        "billing",
        "shipping",
        "other"
      ],
      "title": "Category",
      "type": "string"
    },
    "priority": {
      "maximum": 5,
      "minimum": 1,
      "title": "Priority",
      "type": "integer"
    }
  },
  "required": [
    "category",
    "priority"
  ],
  "title": "Triage",
  "type": "object"
}
